# Stage-1 VAE reconstruction diagnostic v1

This unexecuted Track B notebook diagnoses an existing checkpoint and patch bank. It does not train, alter model capacity or losses, rebuild the patch bank, select a best checkpoint, or start Stage 2. The fixed samples are development diagnostics and are not held out or confirmatory.

In [ ]:
import shutil
import subprocess

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA runtime found. In Colab choose Runtime > Change runtime type > Hardware accelerator > GPU, reconnect, and rerun this cell.")
subprocess.run(["nvidia-smi"], check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access CUDA. Select a Colab GPU runtime, reconnect, and rerun before cloning or installing FieldBridge.")
print(f"CUDA preflight passed: {torch.cuda.get_device_name(0)}")

In [ ]:
from pathlib import Path
import importlib
import importlib.util
import re
import subprocess
import sys

from google.colab import drive

drive.mount("/content/drive")
REPOSITORY_URL = "https://github.com/GuillermoTafoya/MRIxFields.git"
REPO_DIR = Path("/content/FieldBridge-stage1-diagnostic-v1")
EXPECTED_CODE_COMMIT = input("Diagnostic branch commit SHA: " ).strip()
EXPECTED_TRAINING_COMMIT = input("Historical training/checkpoint commit SHA: " ).strip()
CHECKPOINT_PATH = Path(input("External Stage-1 checkpoint path: " ).strip()).expanduser()
PATCH_BANK_DIR = Path(input("External Stage-1 patch-bank directory: " ).strip()).expanduser()
OFFICIAL_MANIFEST_PATH = Path(input("External official JSONL manifest path: " ).strip()).expanduser()
RESOLVED_RUN_CONFIG_PATH = Path("/content/stage1_resolved_training_config.yaml")
OUTPUT_DIR = Path(input("New diagnostic output directory: " ).strip()).expanduser()
SWEEP_TEXT = input("Optional comma-separated checkpoint paths for chronological patch diagnostics: " ).strip()
CHECKPOINT_SWEEP_PATHS = tuple(Path(value.strip()).expanduser() for value in SWEEP_TEXT.split(",") if value.strip())

if re.fullmatch(r"[0-9a-f]{40}", EXPECTED_CODE_COMMIT) is None:
    raise ValueError("Enter the exact 40-character diagnostic branch commit SHA.")
if re.fullmatch(r"[0-9a-f]{40}", EXPECTED_TRAINING_COMMIT) is None:
    raise ValueError("Enter the exact 40-character historical training commit SHA.")
if REPO_DIR.exists() and not (REPO_DIR / ".git").is_dir():
    raise RuntimeError(f"Existing checkout path is not a Git repository: {REPO_DIR}")
if OUTPUT_DIR.exists():
    raise FileExistsError("Use a new output directory.")

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPOSITORY_URL, str(REPO_DIR)], check=True)
else:
    EXISTING_REMOTE = subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True).strip()
    if EXISTING_REMOTE.rstrip("/").removesuffix(".git") != REPOSITORY_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError(f"Existing checkout has unexpected origin: {EXISTING_REMOTE}")
    if subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True).strip():
        raise RuntimeError("Existing checkout has local changes; preserve or remove them before rerunning setup.")
subprocess.run(["git", "fetch", "origin", EXPECTED_CODE_COMMIT], cwd=REPO_DIR, check=True)
subprocess.run(["git", "checkout", "--detach", EXPECTED_CODE_COMMIT], cwd=REPO_DIR, check=True)
ACTUAL_CODE_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if ACTUAL_CODE_COMMIT != EXPECTED_CODE_COMMIT:
    raise RuntimeError(f"Checked out {ACTUAL_CODE_COMMIT}, expected {EXPECTED_CODE_COMMIT}.")
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[dev,nifti,evaluation,perceptual]"], cwd=REPO_DIR, check=True)

SOURCE_DIR = str(REPO_DIR / "src")
if SOURCE_DIR in sys.path:
    sys.path.remove(SOURCE_DIR)
sys.path.insert(0, SOURCE_DIR)
for module_name in tuple(sys.modules):
    if module_name == "fieldbridge" or module_name.startswith("fieldbridge."):
        del sys.modules[module_name]
importlib.invalidate_caches()
import fieldbridge as installed_fieldbridge

PACKAGE_FILE = Path(installed_fieldbridge.__file__).resolve()
if REPO_DIR not in PACKAGE_FILE.parents:
    raise RuntimeError(f"Same-kernel import did not resolve to this checkout: {PACKAGE_FILE}")
print(f"Verified FieldBridge package path: {PACKAGE_FILE}")

In [ ]:
RUNNER_PATH = REPO_DIR / "notebooks" / "stage1_vae_reconstruction_diagnostic_runner.py"
runner_spec = importlib.util.spec_from_file_location("stage1_diagnostic_runner", RUNNER_PATH)
if runner_spec is None or runner_spec.loader is None:
    raise RuntimeError("Unable to load the checked-in diagnostic runner.")
runner = importlib.util.module_from_spec(runner_spec)
runner_spec.loader.exec_module(runner)

RESOLVED_RUN_CONFIG_PATH = runner.reconstruct_resolved_training_yaml(
    repo_dir=REPO_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    patch_bank_dir=PATCH_BANK_DIR,
    expected_training_commit=EXPECTED_TRAINING_COMMIT,
    output_path=RESOLVED_RUN_CONFIG_PATH,
)
print(f"Reconstructed and validated historical training config: {RESOLVED_RUN_CONFIG_PATH}")

HANDOFF = runner.run_stage1_diagnostic(
    repo_dir=REPO_DIR,
    checkpoint_path=CHECKPOINT_PATH,
    patch_bank_dir=PATCH_BANK_DIR,
    official_manifest_path=OFFICIAL_MANIFEST_PATH,
    resolved_run_config_path=RESOLVED_RUN_CONFIG_PATH,
    output_dir=OUTPUT_DIR,
    code_commit=ACTUAL_CODE_COMMIT,
    checkpoint_sweep_paths=CHECKPOINT_SWEEP_PATHS,
)

## Stop after diagnosis

Use `HANDOFF['recommendation']` as the result-dependent engineering recommendation. Do not train another VAE, rebuild the bank, choose a checkpoint post hoc, or start Stage 2 from this notebook.